In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
display(spark.sql("SELECT * FROM bronze_games LIMIT 5"))

# check total rows
display(spark.sql("""
SELECT 
  COUNT(*) as total_rows,
  COUNT(DISTINCT name) as unique_games
FROM bronze_games
"""))

# check duplicates
display(spark.sql("""
SELECT name, COUNT(*) as count 
FROM bronze_games 
GROUP BY name 
HAVING COUNT(*) > 1
"""))

Databricks data profile. Run in Databricks to view.

In [0]:
%sql
CREATE OR REPLACE TABLE Silver_games AS
SELECT 
    name,
    short_description,
    long_description,
    genres,
    minimum_system_requirement,
    recommend_system_requirement,
    COALESCE(
      try_to_date(release_date, 'd MMM, yyyy'),
      try_to_date(release_date, 'MMM yyyy')
    ) as release_date,
    developer,
    publisher,
    overall_player_rating,
    CASE 
    WHEN number_of_reviews_from_purchased_people LIKE '%of%'
        THEN TRY_CAST(
        TRY_CAST(REGEXP_EXTRACT(number_of_reviews_from_purchased_people, '(\\d+)') AS FLOAT) / 100 *
        TRY_CAST(REPLACE(REGEXP_EXTRACT(number_of_reviews_from_purchased_people, 'of ([\\d,]+)'), ',', '') AS INT)
        AS INT)
    ELSE TRY_CAST(
        REGEXP_EXTRACT(REPLACE(number_of_reviews_from_purchased_people, ',', ''), '(\\d+)') AS INT
    )
    END as reviews_purchased,
    CAST(REPLACE(number_of_english_reviews, ',', '') AS INT) as reviews_english,
    link
FROM bronze_games

In [0]:
%sql
SELECT count(*)
FROM Silver_games;

In [0]:
# check null values for important columns
display(spark.sql("""SELECT
  COUNT(*) - COUNT(name) as null_name,
  COUNT(*) - COUNT(release_date) as null_release_date,
  COUNT(*) - COUNT(reviews_purchased) as null_reviews_purchased,
  COUNT(*) - COUNT(overall_player_rating) as null_rating
FROM silver_games"""))

display(spark.sql("""
SELECT name, reviews_purchased
FROM silver_games
WHERE reviews_purchased IS NULL
"""))

In [0]:
%sql
CREATE OR REPLACE TABLE silver_games AS
SELECT *
FROM silver_games
WHERE reviews_purchased IS NOT NULL 
  AND reviews_purchased > 0

In [0]:
%sql
SELECT COUNT(*)
FROM silver_games